In [0]:
-- ===================================
-- DIMENSÕES
-- ===================================

CREATE OR REPLACE TABLE fintech.gold.dim_cliente AS 

SELECT 
  id_cliente,
  nome,
  sexo,
  idade,
  cidade,
  estado,
  data_cadastro,
  faixa_renda,
  score_credito
FROM fintech.silver.clientes

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE fintech.gold.dim_produto AS
SELECT
    id_produto,
    nome_produto,
    categoria
FROM fintech.silver.produtos


num_affected_rows,num_inserted_rows


In [0]:
-- ===============================
-- FATOS
-- ===============================



CREATE OR REPLACE TABLE fintech.gold.fato_transacoes AS

SELECT
    id_transacao,
    id_cliente,
    data,
    tipo,
    valor
FROM fintech.silver.transacoes

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE fintech.gold.fato_contratos AS
SELECT
    id_contrato,
    id_cliente,
    id_produto,
    data_adesao,
    status
FROM fintech.silver.contratos;

num_affected_rows,num_inserted_rows


In [0]:
SHOW TABLES IN fintech.gold

database,tableName,isTemporary
gold,dim_cliente,false
gold,dim_produto,false
gold,fato_contratos,false
gold,fato_transacoes,false


In [0]:
SELECT 
    MIN(data) AS data_minima,
    MAX(data) AS data_maxima
FROM fintech.gold.fato_transacoes


data_minima,data_maxima
2023-01-01,2025-06-19


In [0]:
-- ==============================
-- DIMENSÃO CALENDÁRIO
-- ==============================

CREATE OR REPLACE TABLE fintech.gold.dim_data AS

SELECT
    data,
    YEAR(data) AS ano,
    MONTH(data) AS numero_mes,
    MONTHNAME(data) AS nome_mes,
    QUARTER(data) AS trimestre,
    WEEKOFYEAR(data) AS semana_ano,
    DAY(data) AS dia,
    DAYOFWEEK(data) AS dia_semana_numero
FROM (
    SELECT
        explode( /*tranformando as sequências em linhas */
            sequence( /*transformando as datas em sequência*/
                DATE('2023-01-01'),
                DATE('2025-12-31'),
                interval 1 day
            )
        ) AS data
);

num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM fintech.gold.dim_data
LIMIT 10

data,ano,numero_mes,nome_mes,trimestre,semana_ano,dia,dia_semana_numero
2023-01-01,2023,1,Jan,1,52,1,1
2023-01-02,2023,1,Jan,1,1,2,2
2023-01-03,2023,1,Jan,1,1,3,3
2023-01-04,2023,1,Jan,1,1,4,4
2023-01-05,2023,1,Jan,1,1,5,5
2023-01-06,2023,1,Jan,1,1,6,6
2023-01-07,2023,1,Jan,1,1,7,7
2023-01-08,2023,1,Jan,1,1,8,1
2023-01-09,2023,1,Jan,1,2,9,2
2023-01-10,2023,1,Jan,1,2,10,3


In [0]:
SELECT COUNT(*)
FROM fintech.gold.dim_data

COUNT(*)
1096


In [0]:
CREATE OR REPLACE TABLE fintech.gold.dim_data AS

SELECT
    data,

    YEAR(data) AS ano,

    MONTH(data) AS numero_mes,

    CONCAT(
        YEAR(data),
        '-',
        LPAD(MONTH(data),2,'0')
    ) AS ano_mes,

    MONTHNAME(data) AS nome_mes,

    QUARTER(data) AS trimestre,

    WEEKOFYEAR(data) AS semana_ano,

    DAY(data) AS dia

FROM (
    SELECT
        explode(
            sequence(
                DATE('2023-01-01'),
                DATE('2025-12-31'),
                interval 1 day
            )
        ) AS data
);

num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM fintech.gold.dim_data
LIMIT 10

data,ano,numero_mes,ano_mes,nome_mes,trimestre,semana_ano,dia
2023-01-01,2023,1,2023-01,Jan,1,52,1
2023-01-02,2023,1,2023-01,Jan,1,1,2
2023-01-03,2023,1,2023-01,Jan,1,1,3
2023-01-04,2023,1,2023-01,Jan,1,1,4
2023-01-05,2023,1,2023-01,Jan,1,1,5
2023-01-06,2023,1,2023-01,Jan,1,1,6
2023-01-07,2023,1,2023-01,Jan,1,1,7
2023-01-08,2023,1,2023-01,Jan,1,1,8
2023-01-09,2023,1,2023-01,Jan,1,2,9
2023-01-10,2023,1,2023-01,Jan,1,2,10


In [0]:
-- ========================
-- TABELA ANALÍTICA
-- ========================

CREATE OR REPLACE TABLE fintech.gold.dashboard_executivo AS
SELECT
    t.data,
    COUNT(DISTINCT t.id_cliente) AS clientes_ativos,
    COUNT(t.id_transacao) AS quantidade_transacoes,
    SUM(t.valor) AS receita_total,
    AVG(t.valor) AS tiket_medio
FROM fintech.gold.fato_transacoes t
GROUP BY t.data;

num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM fintech.gold.dashboard_executivo
LIMIT 20

data,clientes_ativos,quantidade_transacoes,receita_total,tiket_medio
2023-02-11,24,24,5723.30,238.470833
2025-02-18,32,32,8150.37,254.699063
2024-06-30,26,26,4500.76,173.106154
2024-04-11,24,24,6722.69,280.112083
2023-08-27,19,19,6003.00,315.947368
2025-04-11,23,23,5628.29,244.708261
2023-05-05,16,17,4486.10,263.888235
2025-02-08,32,32,7686.04,240.188750
2024-05-13,17,17,4994.49,293.793529
2024-02-22,25,25,5871.43,234.857200


In [0]:
SELECT *
FROM fintech.gold.dashboard_executivo
LIMIT 20

data,clientes_ativos,quantidade_transacoes,receita_total,tiket_medio
2023-02-11,24,24,5723.30,238.470833
2025-02-18,32,32,8150.37,254.699063
2024-06-30,26,26,4500.76,173.106154
2024-04-11,24,24,6722.69,280.112083
2023-08-27,19,19,6003.00,315.947368
2025-04-11,23,23,5628.29,244.708261
2023-05-05,16,17,4486.10,263.888235
2025-02-08,32,32,7686.04,240.188750
2024-05-13,17,17,4994.49,293.793529
2024-02-22,25,25,5871.43,234.857200


In [0]:
/*KPIs de Receita Total, Quantidade de Transações e Ticket Médio*/
SELECT
    SUM(valor) AS receita_total,
    COUNT(*) AS quantidade_transacoes,
    AVG(valor) AS ticket_medio
FROM fintech.gold.fato_transacoes;

receita_total,quantidade_transacoes,ticket_medio
5167602.04,20000,258.380102


In [0]:
/*KPIs de Total de Clientes*/
SELECT
    COUNT(id_cliente) AS total_cliente
FROM fintech.gold.dim_cliente;

/*SELECT
    estado,
    COUNT(*) AS clientes
FROM fintech.gold.dim_cliente
GROUP BY estado
ORDER BY clientes DESC*/

total_cliente,estado,clientes
216,RS,216
209,SP,209
201,RJ,201
197,PR,197
177,MG,177


In [0]:
/*KPI de Clientes por estado*/
SELECT
    estado,
    COUNT(*) AS clientes
FROM fintech.gold.dim_cliente
GROUP BY estado
ORDER BY clientes DESC

In [0]:
/*KPI de Contratos por Produto*/

SELECT
    p.nome_produto,
    COUNT(*) AS contratos
FROM fintech.gold.fato_contratos c
INNER JOIN fintech.gold.dim_produto p 
    ON c.id_produto = p.id_produto
GROUP BY p.nome_produto
ORDER BY contratos DESC;


nome_produto,contratos
Empréstimo Pessoal,530
Cartão Gold,529
Cartão Platinum,492
Conta Digital,482
Investimentos,467


In [0]:
SELECT COUNT(*) FROM fintech.gold.dim_cliente

COUNT(*)
1000


In [0]:
SELECT COUNT (*) FROM fintech.gold.dim_produto

COUNT(*)
5


In [0]:
SELECT COUNT(*) FROM fintech.gold.dim_data

COUNT(*)
1096


In [0]:
SELECT COUNT(*) FROM fintech.gold.fato_transacoes

COUNT(*)
20000


In [0]:
SELECT COUNT(*) FROM fintech.gold.fato_contratos;

COUNT(*)
2500


In [0]:
SHOW TABLES IN fintech.gold;

database,tableName,isTemporary
gold,dashboard_executivo,false
gold,dim_cliente,false
gold,dim_data,false
gold,dim_produto,false
gold,fato_contratos,false
gold,fato_transacoes,false
